# Task 8 — Color and physical-feature Stage C

Extract RGB/Lab correlation, PRNU-coherence, and chromatic-aberration diagnostics. Train only the color families on matched-Q96 views; physical families require eligible source-original data. A GPU is not required.

In [1]:
# 1. Mount Drive, update the disposable checkout, and install the project.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

PROJECT_ROOT = Path('/content/cya-techjam26')
REPOSITORY_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if (PROJECT_ROOT / '.git').is_dir():
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.splitlines()
    unexpected = [line for line in status if not line.endswith('configs/colab.json')]
    assert not unexpected, f'Unexpected checkout changes: {unexpected}'
    if status:
        subprocess.run(['git', 'restore', 'configs/colab.json'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=PROJECT_ROOT, check=True)

Mounted at /content/drive


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '.', '--no-deps'], returncode=0)

In [2]:
# 2. Restore the fixed-Q96 inputs and durable Task 8 artifacts.
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
input_archive = DRIVE_ARTIFACT_ROOT / 'task2_stagea_bundle.tar.gz'
assert input_archive.is_file(), input_archive
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(input_archive, TASK2_ROOT)
manifest = TASK2_ROOT / 'fixed_q96_manifest.csv'
assert manifest.is_file(), manifest
TASK8_ROOT = ARTIFACT_ROOT / 'task8'
DRIVE_TASK8_ROOT = DRIVE_ARTIFACT_ROOT / 'task8'
if DRIVE_TASK8_ROOT.is_dir():
    shutil.copytree(DRIVE_TASK8_ROOT, TASK8_ROOT, dirs_exist_ok=True)
print('Task 8 inputs ready')

Task 8 inputs ready


In [3]:
# 3. Extract once; reuse only the current versioned configuration.
feature_table = TASK8_ROOT / 'auxiliary_features.csv'
extraction_report = TASK8_ROOT / 'extraction_report.json'
auxiliary_config = json.loads((PROJECT_ROOT / 'configs/colab.json').read_text())['auxiliary']
saved_report = json.loads(extraction_report.read_text()) if extraction_report.is_file() else {}
reuse = feature_table.is_file() and saved_report.get('configuration') == auxiliary_config and saved_report.get('final_test_read') is False
if reuse:
    print('SKIP complete auxiliary extraction')
else:
    subprocess.run([
        sys.executable, 'scripts/extract_auxiliary_features.py',
        '--manifest', str(manifest), '--output', str(feature_table),
        '--report', str(extraction_report),
        '--cache-root', '/content/auxiliary_feature_cache',
        '--matching-policy', 'fixed_q96',
    ], cwd=PROJECT_ROOT, check=True)
    DRIVE_TASK8_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(feature_table, DRIVE_TASK8_ROOT / feature_table.name)
    shutil.copy2(extraction_report, DRIVE_TASK8_ROOT / extraction_report.name)
report = json.loads(extraction_report.read_text())
print('Rows/features:', report['valid_count'], report['feature_count'])
print('Physical warning:', report['physical_claim_warning'])
print('Radial distortion:', report['radial_distortion_status'])
print('Eligibility/validity/confidence audit:')
for row in report['family_audit']:
    print(row)

Rows/features: 1390 67
Physical warning: PRNU is a single-image coherence proxy; no camera or lens authenticity claim is made.
Radial distortion: deferred_until_eligible_line_support_is_available
Eligibility/validity/confidence audit:
{'confidence_mean': 0.9967051070840197, 'eligibility_rate': 1.0, 'eligible_count': 607, 'family': 'rgb', 'label': 'authentic', 'row_count': 607, 'split': 'seed_train', 'valid_count': 607, 'validity_rate': 1.0}
{'confidence_mean': 0.9909060955518942, 'eligibility_rate': 1.0, 'eligible_count': 607, 'family': 'lab', 'label': 'authentic', 'row_count': 607, 'split': 'seed_train', 'valid_count': 607, 'validity_rate': 1.0}
{'confidence_mean': 1.0, 'eligibility_rate': 0.0, 'eligible_count': 0, 'family': 'prnu', 'label': 'authentic', 'row_count': 607, 'split': 'seed_train', 'valid_count': 607, 'validity_rate': 1.0}
{'confidence_mean': 0.00957874498219834, 'eligibility_rate': 0.0, 'eligible_count': 0, 'family': 'ca', 'label': 'authentic', 'row_count': 607, 'split':

In [4]:
# 4. Train RGB, Lab, and combined color baselines across three seeds.
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

feature_sha = sha256_file(feature_table)
variants = ('rgb', 'lab', 'rgb_lab')
seeds = (42, 43, 44)
for variant in variants:
    for seed in seeds:
        local_run = TASK8_ROOT / variant / f'seed_{seed}'
        drive_run = DRIVE_TASK8_ROOT / variant / f'seed_{seed}'
        run_report = local_run / 'report.json'
        saved = json.loads(run_report.read_text()) if run_report.is_file() else {}
        if saved.get('feature_table_sha256') == feature_sha:
            print(f'SKIP complete: {variant}, seed {seed}')
            continue
        print(f'RUN: {variant}, seed {seed}')
        subprocess.run([
            sys.executable, 'scripts/train_auxiliary_baseline.py',
            '--features', str(feature_table), '--output', str(local_run),
            '--variant', variant, '--seed', str(seed),
        ], cwd=PROJECT_ROOT, check=True)
        shutil.copytree(local_run, drive_run, dirs_exist_ok=True)

RUN: rgb, seed 42
RUN: rgb, seed 43
RUN: rgb, seed 44
RUN: lab, seed 42
RUN: lab, seed 43
RUN: lab, seed 44
RUN: rgb_lab, seed 42
RUN: rgb_lab, seed 43
RUN: rgb_lab, seed 44


In [5]:
# 5. Select the clean color representation. Physical-family retention stays pending.
comparison_path = TASK8_ROOT / 'color_comparison.json'
subprocess.run([
    sys.executable, 'scripts/compare_color_variants.py',
    '--task8-root', str(TASK8_ROOT), '--output', str(comparison_path),
], cwd=PROJECT_ROOT, check=True)
shutil.copy2(comparison_path, DRIVE_TASK8_ROOT / comparison_path.name)
comparison = json.loads(comparison_path.read_text())
print('Selected color representation:', comparison['selected_color_representation'])
print(json.dumps(comparison['accuracy_mean'], indent=2))
print('PRNU/CA retention:', comparison['physical_family_retention'])
print('No PRNU or optics authenticity claim has been made.')

Selected color representation: lab
{
  "lab": 0.8242424242424242,
  "rgb": 0.5515151515151515,
  "rgb_lab": 0.7878787878787877
}
PRNU/CA retention: pending_eligible_data_and_task3
No PRNU or optics authenticity claim has been made.
